# 📖 Designing Data-Intensive Applications
## Bölüm 6: Replication (Çoğaltma)

> *"Yanlış gidebilecek bir şey ile kesinlikle yanlış gidemeyecek bir şey arasındaki temel fark şudur: Kesinlikle yanlış gidemeyecek bir şey yanlış gittiğinde, genellikle ona ulaşmak veya onarmak imkânsız olduğu ortaya çıkar."*
> 
> — Douglas Adams, *Mostly Harmless* (1992)

---

## 📋 İçindekiler

1. [Replication Nedir ve Neden Gereklidir?](#1-replication-nedir)
2. [Backups vs Replication](#2-backups-vs-replication)
3. [Single-Leader Replication](#3-single-leader-replication)
4. [Synchronous vs Asynchronous Replication](#4-synchronous-vs-asynchronous)
5. [Yeni Follower Kurulumu (Setting Up New Followers)](#5-yeni-follower-kurulumu)
6. [Object Storage Destekli Veritabanları](#6-object-storage)
7. [Node Outage'larını Yönetmek](#7-node-outages)
8. [Replication Log Implementasyonları](#8-replication-log-impl)
9. [Replication Lag Sorunları](#9-replication-lag)
10. [Multi-Leader Replication](#10-multi-leader-replication)
11. [Sync Engines ve Local-First Software](#11-sync-engines)
12. [Conflicting Write'larla Başa Çıkma](#12-conflict-resolution)
13. [Leaderless Replication](#13-leaderless-replication)
14. [Quorum ile Okuma/Yazma](#14-quorum)
15. [Concurrent Write Tespiti](#15-concurrent-writes)
16. [Özet ve Karşılaştırma](#16-ozet)

---
## 1. Replication Nedir ve Neden Gereklidir? <a id='1-replication-nedir'></a>

**Replication** (çoğaltma), aynı verinin kopyasını bir ağ üzerinden birbirine bağlı birden fazla makinede tutmak anlamına gelir.

### Replication Neden İstenir?

Verinin replike edilmesini istemenizin birkaç temel nedeni vardır:

1. **Coğrafi Yakınlık (Geographic Proximity):** Verinin kullanıcılarınıza coğrafi olarak yakın tutulması, erişim gecikmesini (access latency) azaltır. Örneğin, Avrupa'daki kullanıcılar için Avrupa'da bir replica tutmak, her okuma isteğinin ABD'deki sunucuya gitmesini engeller.

2. **High Availability (Yüksek Erişilebilirlik):** Sistemin bazı parçaları arızalansa bile çalışmaya devam etmesine izin verir. Bir node çökerse, başka bir node devralabilir.

3. **Read Throughput (Okuma Kapasitesi):** Read sorgularına hizmet verebilen makine sayısını artırır. Birden fazla replica, okuma isteklerini dağıtabilir ve tek bir sunucunun üzerindeki yükü azaltır.

### Temel Kısıt

Bu bölümde, **dataset'inizin her makinenin tüm veriyi tutabileceği kadar küçük** olduğunu varsayıyoruz. Çok büyük dataset'ler için *sharding* (bölümleme) gerekir; bu konu 7. bölümde ele alınacaktır.

### Replikasyonun Gerçek Zorluğu Nedir?

Eğer replike ettiğiniz veri hiç değişmeseydi, replikasyon son derece kolay olurdu: veriyi her node'a bir kez kopyalarsınız, iş biter. **Asıl zorluk, replikalara yapılan değişiklikleri yönetmektir.**

Bu bölümde node'lar arasında değişiklikleri replike etmek için üç temel algoritma ailesi ele alınacaktır:

| Algoritma | Açıklama |
|-----------|----------|
| **Single-leader** | Tüm yazımlar tek bir node'a (leader) gider; diğerleri (followers) değişiklikleri alır |
| **Multi-leader** | Birden fazla node yazım kabul edebilir; bunlar birbirleriyle senkronize olur |
| **Leaderless** | Herhangi bir replica yazım kabul edebilir; lider kavramı yoktur |

Neredeyse tüm distributed veritabanları bu üç yaklaşımdan birini kullanır. Her birinin artıları ve eksileri vardır.

### Trade-off'lar

Replikasyonda birçok **trade-off** (değiş tokuş) göz önünde bulundurulmalıdır. Örneğin:
- **Synchronous (senkron) mu yoksa asynchronous (asenkron) mu** replikasyon kullanılacak?
- Arızalı replica'lar nasıl yönetilecek?

Bu seçimler genellikle veritabanı yapılandırma seçenekleridir; ayrıntılar veritabanına göre değişse de genel ilkeler birçok implementasyonda benzerdir.

### Tarihsel Not

Veritabanı replikasyonu eski bir konudur. İlkeler 1970'lerde incelendiğinden bu yana çok değişmemiştir; bunun nedeni, ağların temel kısıtlarının aynı kalmış olmasıdır. Yine de **eventual consistency** (nihai tutarlılık) gibi kavramlar hâlâ kafa karışıklığına yol açmaktadır.

---
## 2. Backups vs Replication (Yedekler ve Replikasyon) <a id='2-backups-vs-replication'></a>

### Hâlâ Backup'a İhtiyacınız Var mı?

Replikasyon varken **backup'a hâlâ ihtiyaç duyulduğunu** merak edebilirsiniz. Cevap **evet**'tir; çünkü farklı amaçlara hizmet ederler:

- **Replica'lar:** Bir node'daki yazımları diğer node'lara hızla yansıtır. Anlık görüntüler değil, *canlı* kopyalardır.
- **Backup'lar:** Verinin eski anlık görüntülerini (old snapshots) saklar, böylece zamanda geriye gidebilirsiniz.

### Neden Sadece Replication Yeterli Değil?

Bazı verileri yanlışlıkla silerseniz, replikasyon yardımcı olmaz; çünkü **silme işlemi de tüm replicalara yayılacaktır**. Silinen veriyi geri yüklemek istiyorsanız backup'a ihtiyaç duyarsınız.

### Birbirini Tamamlayan Kavramlar

Replikasyon ve backup çoğunlukla birbirini tamamlar:
- Backup'lar bazen yeni bir follower kurma sürecinin parçasıdır.
- Replikasyon log'larını arşivlemek, bir backup sürecinin parçası olabilir.

### Immutable Snapshots (Değişmez Anlık Görüntüler)

Bazı veritabanları, bir çeşit iç yedekleme görevi gören **geçmiş durumların değişmez anlık görüntülerini** dahili olarak saklar. Ancak bu, verinin eski sürümlerini mevcut durumla aynı depolama ortamında tutmak anlamına gelir.

Büyük miktarda veriniz varsa, eski verinin yedeklerini seyrek erişilen veriler için optimize edilmiş bir **object store**'da tutmak ve veritabanının yalnızca mevcut durumunu primary storage'da saklamak daha ekonomik olabilir.

---
## 3. Single-Leader Replication <a id='3-single-leader-replication'></a>

### Replica Nedir?

Veritabanının bir kopyasını saklayan her node'a **replica** denir. Birden fazla replica olduğunda, kaçınılmaz olarak şu soru ortaya çıkar: Tüm verinin tüm replicalara ulaşmasını nasıl sağlarız?

Veritabanına yapılan her yazımın her replica tarafından işlenmesi gerekir; aksi takdirde replikalar artık aynı veriyi içermez.

### Leader-Based Replication Nasıl Çalışır?

En yaygın çözüm **leader-based** (lider tabanlı), **primary-backup** ya da **active/passive** replication olarak adlandırılır. Çalışma prensibi şöyledir:

1. **Leader (Lider) Seçimi:** Replikalardan biri **leader** (aynı zamanda *primary* veya *source* olarak da bilinir) olarak belirlenir. İstemciler veritabanına yazım yapmak istediklerinde isteklerini leader'a göndermek zorundadır; leader önce yeni veriyi kendi yerel depolama alanına yazar.

2. **Follower'lara Dağıtım:** Diğer replikalar **follower** (aynı zamanda *read replicas*, *secondaries* veya *hot standbys* olarak da bilinir) olarak tanımlanır. Leader yerel depolama alanına yeni veri yazdığında, veri değişikliğini **replication log** veya **change stream** adı verilen bir akışın parçası olarak tüm follower'lara gönderir. Her follower, bu log'u leader'dan alır ve tüm yazımları leader'da işlendikleri sırayla uygulayarak veritabanının kendi yerel kopyasını günceller.

3. **Okuma İşlemleri:** Bir istemci veritabanından okumak istediğinde, hem leader'ı hem de herhangi bir follower'ı sorgulayabilir. Ancak **yazımlar yalnızca leader tarafından kabul edilir** (follower'lar istemci açısından salt okunurdur).

```
┌─────────────────────────────────────────────────────────┐
│                     CLIENT                              │
│                        │                               │
│                    YAZIM (Write)                        │
│                        ↓                               │
│              ┌─────────────────┐                        │
│              │    LEADER       │  ← Tüm yazımlar buraya │
│              └────────┬────────┘                        │
│                 replication log                         │
│              ┌────────┴────────┐                        │
│              ↓                 ↓                        │
│       ┌──────────┐      ┌──────────┐                    │
│       │FOLLOWER 1│      │FOLLOWER 2│  ← Okumalar buraya │
│       └──────────┘      └──────────┘                    │
└─────────────────────────────────────────────────────────┘
```

### Hangi Sistemler Single-Leader Replication Kullanır?

Single-leader replication son derece yaygın kullanılmaktadır:

- **İlişkisel veritabanları:** PostgreSQL, MySQL, Oracle Data Guard, SQL Server Always On
- **Doküman veritabanları:** MongoDB, DynamoDB
- **Message broker'lar:** Kafka
- **Dağıtık blok cihazlar:** DRBD
- **Consensus algoritmaları:** Raft (CockroachDB, TiDB, etcd, RabbitMQ quorum queues'da kullanılır)

### Sharding ile İlişkisi

Veritabanı sharded (bölünmüş) ise her shard'ın bir leader'ı vardır. Farklı shard'ların farklı node'larda leader'ları olabilir, ancak her shard'ın tek bir leader node'u olmalıdır.

### Terminoloji Notu

Eski belgelerde **master–slave replication** terimiyle karşılaşabilirsiniz. Bu, leader-based replication ile aynı şeyi ifade eder, ancak bu terim artık yaygın olarak rahatsız edici kabul edildiğinden kullanımından kaçınılmalıdır.

---
## 4. Synchronous vs Asynchronous Replication <a id='4-synchronous-vs-asynchronous'></a>

Replike edilmiş bir sistemin önemli bir ayrıntısı, replikasyonun **synchronous (senkron)** mu yoksa **asynchronous (asenkron)** mu gerçekleştiğidir. (İlişkisel veritabanlarında bu genellikle yapılandırılabilir bir seçenektir; diğer sistemler ise çoğunlukla birinden veya diğerinden olacak şekilde sabit kodlanmıştır.)

### Senaryo: Profil Resmi Güncelleme

Bir web sitesi kullanıcısının profil resmini güncellediğini düşünelim:

1. İstemci güncelleme isteğini leader'a gönderir
2. Leader isteği alır, veri değişikliğini follower'lara iletir
3. Leader, güncellemenin başarılı olduğunu istemciye bildirir

### Synchronous Replication

**Synchronous (senkron)** replikasyonda, leader, başarıyı kullanıcıya raporlamadan ve yazmayı diğer istemcilere görünür kılmadan önce follower'ın yazmayı aldığını onaylamasını **bekler**.

```
Client → Leader → [BEKLE] → Follower 1 onay gönderir → Client'a başarı bildirimi
```

**Avantaj:** Follower'ın leader ile tutarlı, güncel bir veri kopyasına sahip olması garanti edilir. Leader aniden arızalanırsa verinin follower'da hâlâ mevcut olduğundan emin olunabilir.

**Dezavantaj:** Senkron follower yanıt vermezse (çöktüğü için, ağ arızası nedeniyle veya başka bir nedenle), yazma işlemi gerçekleştirilemez. Leader, senkron replica tekrar kullanılabilir hale gelene kadar tüm yazımları engellemelidir.

### Asynchronous Replication

**Asynchronous (asenkron, nonblocking)** replikasyonda, leader mesajı gönderir ancak follower'dan yanıt beklemez.

```
Client → Leader → Client'a hemen başarı bildirimi
Leader ... (sonra) ... Follower 2'ye gönderir
```

Normalde replikasyon oldukça hızlıdır; çoğu veritabanı sistemi değişiklikleri follower'lara bir saniyeden kısa sürede uygular. Ancak ne kadar süreceğine dair bir **garanti yoktur**. Bazı durumlarda follower'lar leader'ın birkaç dakika ya da daha fazla gerisinde kalabilir; örneğin bir follower arızadan kurtarılıyorsa, sistem maksimum kapasitede çalışıyorsa veya node'lar arasında ağ sorunları varsa.

### Semisynchronous Replication

Tüm follower'ların senkron olması pratik değildir; herhangi bir node kesintisi tüm sistemi durdurur. Bu nedenle, bir veritabanı senkron replikasyon sunuyorsa, bu genellikle follower'lardan **birinin** senkron, diğerlerinin ise asenkron olduğu anlamına gelir.

Senkron follower kullanılamaz hale gelirse veya yavaşlarsa, asenkron follower'lardan biri senkron yapılır. Bu, en az iki node'da güncel bir veri kopyasının bulunmasını garanti eder: leader ve bir senkron follower. Bu yapılandırmaya **semisynchronous** (yarı senkron) da denir.

### Quorum ile Çoğunluk Senkronizasyonu

Bazı sistemlerde, replika'ların bir **çoğunluğu** (örneğin beş replikanın üçü, leader dahil) senkron olarak güncellenir ve geri kalan azınlık asenkrondur. Bu bir **quorum** örneğidir.

### Tamamen Asenkron Replikasyon

Bazen leader tabanlı replikasyon tamamen asenkron olacak şekilde yapılandırılır. Bu durumda, leader arızalanırsa ve kurtarılamazsa, follower'lara henüz replike edilmemiş yazımlar **kaybolur**. Bu, istemciye onaylanmış olsa bile bir yazımın kalıcı olacağının garanti edilmediği anlamına gelir.

Ancak tamamen asenkron yapılandırmanın avantajı, tüm follower'lar geride kalmış olsa bile leader'ın yazımları işlemeye devam edebilmesidir.

**Dayanıklılığı (durability) zayıflatmak** kötü bir trade-off gibi görünebilir, ancak asenkron replikasyon yaygın şekilde kullanılmaktadır; özellikle çok sayıda follower varsa veya bunlar coğrafi olarak dağıtılmışsa.

---
## 5. Yeni Follower Kurulumu (Setting Up New Followers) <a id='5-yeni-follower-kurulumu'></a>

Zaman zaman yeni follower'lar kurmanız gerekir; belki replica sayısını artırmak ya da arızalı node'ların yerini doldurmak için. Yeni follower'ın leader'ın verisinin doğru bir kopyasına sahip olmasını nasıl sağlarsınız?

### Neden Sadece Dosya Kopyalamak Yetmez?

Veri dosyalarını bir node'dan diğerine kopyalamak genellikle yeterli değildir. İstemciler sürekli veritabanına yazım yapmaktadır ve veri her zaman değişmektedir; bu nedenle standart bir dosya kopyası, veritabanının farklı bölümlerini farklı zaman noktalarında görür. Sonuç tutarsız olabilir.

Diskteki dosyaları tutarlı hale getirmek için veritabanını kilitleyebilirsiniz (yazımlar için kullanılamaz hale getirebilirsiniz), ancak bu, yüksek erişilebilirlik hedefimizle çelişir.

### Downtime Olmadan Follower Kurma Süreci

Neyse ki, follower kurulumu genellikle downtime olmadan yapılabilir. Kavramsal olarak süreç şöyledir:

**Adım 1 — Consistent Snapshot Alma:**
Mümkünse veritabanının tamamını kilitlemeden, belirli bir zaman noktasındaki tutarlı bir **snapshot** (anlık görüntü) alın. Çoğu veritabanının bu özelliği vardır; backup için de gereklidir. Bazı durumlarda MySQL için Percona XtraBackup gibi üçüncü taraf araçlara ihtiyaç duyulabilir.

**Adım 2 — Snapshot Aktarımı:**
Snapshot'ı yeni follower node'una kopyalayın.

**Adım 3 — Replication Log ile Senkronizasyon:**
Follower, leader'a bağlanır ve snapshot alındığından bu yana gerçekleşen tüm veri değişikliklerini talep eder. Bu, snapshot'ın leader'ın replication log'undaki kesin bir konumla ilişkilendirilmesini gerektirir. Bu konum farklı isimler alır; örneğin:
- PostgreSQL: **log sequence number** (LSN)
- MySQL: **binlog coordinates** veya **global transaction identifiers** (GTIDs)

**Adım 4 — Catch-up (Yetişme):**
Follower, snapshot'tan bu yana gerçekleşen veri değişikliklerinin birikmiş dosyasını (backlog) işlediğinde, **caught up** (yakalamış) olduğu söylenir. Artık gerçek zamanlı olarak leader'dan veri değişikliklerini işlemeye devam edebilir.

### Pratik Adımlar Veritabanına Göre Değişir

Follower kurmanın pratik adımları veritabanına göre önemli ölçüde farklılık gösterir. Bazı sistemlerde süreç tamamen otomatikleştirilmişken, diğerlerinde yönetici tarafından manuel olarak gerçekleştirilmesi gereken çok adımlı karmaşık bir iş akışı olabilir.

### Object Storage ile Replikasyon Log'u Arşivleme

Replication log'unu tüm veritabanının periyodik snapshot'larıyla birlikte bir **object store**'a arşivleyebilirsiniz. Bu, veritabanı backup'larını ve disaster recovery'yi uygulamanın iyi bir yoludur. Yeni bir follower kurmanın 1. ve 2. adımlarını, bu dosyaları object store'dan indirerek gerçekleştirebilirsiniz:

- **WAL-G:** PostgreSQL, MySQL ve SQL Server için bu yaklaşımı uygular
- **Litestream:** SQLite için eşdeğerini yapar

---
## 6. Object Storage Destekli Veritabanları <a id='6-object-storage'></a>

Object storage yalnızca arşivleme amacıyla kullanılmaz. Birçok veritabanı, **canlı sorgulara hizmet vermek** için Amazon S3, Google Cloud Storage ve Azure Blob Storage gibi object store'ları kullanmaya başlamıştır.

### Object Storage'ın Avantajları

**1. Maliyet Etkinliği:** Object storage, diğer bulut depolama seçeneklerine kıyasla ucuzdur. Bu, bulut veritabanlarının daha az sorgulanan verileri daha ucuz, daha yüksek gecikmeli (higher-latency) bir depolamada saklamasına, sık erişilen çalışma setini (working set) ise bellekte, SSD'lerde ve NVMe'de tutmasına olanak tanır.

**2. Yüksek Dayanıklılık:** Object store'lar çok bölgeli (multi-zone, dual-region veya multi-region) replikasyon ve çok yüksek dayanıklılık garantileri sunar. Bu aynı zamanda veritabanlarının bölgeler arası ağ ücretlerinden kaçınmasına da olanak tanır.

**3. Conditional Write ile Transaction ve Leadership Election:** Veritabanları, transaction'ları ve leadership election'ı uygulamak için object store'un **conditional write** özelliğini (esasen bir **compare-and-set**, CAS işlemi) kullanabilir.

**4. Veri Entegrasyonunu Basitleştirme:** Birden fazla veritabanının verilerini aynı object store'da saklamak, özellikle Parquet ve Iceberg gibi açık formatlar kullanıldığında, veri entegrasyonunu basitleştirebilir.

Bu faydalar, transaction'lar, leadership election ve replikasyon sorumluluğunu object storage'a devrederek veritabanı mimarisini önemli ölçüde basitleştirir.

### Object Storage'ın Dezavantajları

Object storage benimseyen sistemler çeşitli trade-off'larla mücadele etmek zorundadır:

- **Yüksek Gecikme (High Latency):** Object store'ların okuma ve yazma gecikmeleri, yerel diskler veya Amazon EBS gibi sanal blok cihazlara kıyasla çok daha yüksektir.
- **API Başına Ücret:** Birçok bulut sağlayıcısı API çağrısı başına ücret talep eder; bu da maliyeti düşürmek için okumaları ve yazımları toplu hale getirmeyi (batch) zorunlu kılar. Bu toplu işlemler gecikmeyi daha da artırır.
- **Immutability (Değişmezlik):** Nesneler genellikle değişmezdir; bu da büyük bir nesnede rastgele yazımları son derece kaynak yoğun bir operasyon yapar.
- **Standart Dosya Sistemi Arayüzü Eksikliği:** Birçok object store, standart dosya sistemi arayüzleri sunmaz. **FUSE** (filesystem in userspace) gibi arayüzler, operatörlerin object store bucket'larını uygulamaların kullanabileceği dosya sistemleri olarak bağlamasına olanak tanır. Ancak birçok FUSE arayüzü, sistemlerin bağımlı olduğu sıralı olmayan yazımlar (non-sequential writes) veya symlink'ler gibi **POSIX** özelliklerinden yoksundur.

### Farklı Mimariler

Farklı sistemler bu trade-off'ları farklı şekillerde ele alır:

- **Tiered Storage (Katmanlı Depolama):** Daha az erişilen verileri object storage'a yerleştirirken, yeni veya sık erişilen verileri SSD, NVMe hatta bellekte tutar.
- **Zero-Disk Architecture (ZDA):** Tüm verileri object storage'a kalıcı hale getirir ve diskleri/belleği yalnızca önbellekleme (caching) için kullanır. Bu, node'ların kalıcı durum içermemesini sağlar ve operasyonları büyük ölçüde basitleştirir. Örnekler: WarpStream, Confluent Freight, Redpanda Serverless, Turbopuffer, SlateDB.

---
## 7. Node Outage'larını Yönetmek (Handling Node Outages) <a id='7-node-outages'></a>

Sistemdeki herhangi bir node beklenmedik şekilde (arıza nedeniyle) veya planlı bakım sırasında (örneğin bir çekirdek güvenlik yamasını kurmak için yeniden başlatma) kapanabilir. Bireysel node'ları downtime olmadan yeniden başlatabilmek, operasyonlar ve bakım açısından büyük bir avantajdır.

Hedefimiz, bireysel node arızalarına rağmen sistemin bir bütün olarak çalışmaya devam etmesini sağlamak ve bir node outage'ının etkisini minimuma indirmektir.

### Follower Arızası: Catch-up Recovery

Her follower, yerel diskinde leader'dan aldığı veri değişikliklerinin bir log'unu tutar. Bir follower çökerse ve yeniden başlatılırsa, ya da leader ile follower arasındaki ağ geçici olarak kesintiye uğrarsa, follower oldukça kolay bir şekilde kurtarılabilir:

Log'undan, arızadan önce işlenen son transaction'ı bilir. Böylece follower leader'a bağlanabilir ve follower bağlantısı kesildiği süre zarfında gerçekleşen tüm veri değişikliklerini talep edebilir. Bu değişiklikler uygulandığında, follower leader'a yetişmiş (caught up) olur ve daha önce olduğu gibi bir veri değişikliği akışı almaya devam edebilir.

**Performans Zorluğu:**
Follower kurtarma kavramsal olarak basit olsa da performans açısından zorlayıcı olabilir. Veritabanının yüksek bir write throughput'u varsa veya follower uzun süre çevrimdışı kaldıysa, yetişilmesi gereken çok sayıda yazım olabilir. Yetişme süreci devam ederken hem kurtarılan follower hem de leader (biriken yazımları follower'a göndermesi gereken) üzerinde yüksek yük oluşacaktır.

**Log Saklama Trade-off'u:**
Leader, tüm follower'ların işlediğini onayladıktan sonra yazım log'unu silebilir. Ancak bir follower uzun süre kullanılamıyorsa, leader iki seçimle karşı karşıya kalır:
- Follower kurtarılıp yetişene kadar log'u saklamak (leader'da disk alanının tükenmesi riski)
- Kullanılamayan follower'ın henüz onaylamadığı log'u silmek (follower geri döndüğünde log'dan kurtarılamaz ve bir backup'tan geri yüklenmesi gerekir)

### Leader Arızası: Failover

Leader'ın arızalanmasını yönetmek daha karmaşıktır. Follower'lardan birinin yeni leader olarak **promote** edilmesi, istemcilerin yazımlarını yeni leader'a gönderecek şekilde yeniden yapılandırılması ve diğer follower'ların yeni leader'dan veri değişikliklerini tüketmeye başlaması gerekir. Bu sürece **failover** denir.

Failover manuel olarak (bir yönetici leader'ın arızalandığına dair bildirim alır ve yeni bir leader oluşturmak için gerekli adımları atar) veya otomatik olarak gerçekleşebilir.

**Otomatik Failover Adımları:**

1. **Leader'ın Arızalandığını Tespit Etme:** Birçok şey yanlış gidebilir: çökmeler, güç kesintileri, ağ sorunları vb. Neyin gerçekleştiğini tespit etmenin kusursuz bir yolu yoktur; bu nedenle çoğu sistem basitçe bir **timeout** kullanır. Node'lar birbirlerine sürekli mesaj gönderir ve bir node belirli bir süre boyunca (örneğin 30 saniye) yanıt vermezse, ölü olduğu varsayılır.

2. **Yeni Leader Seçimi:** Bu, kalan replika'ların çoğunluğu tarafından seçim yoluyla yapılabilir veya yeni bir leader, önceden kurulmuş bir **controller node** tarafından atanabilir. Leadership için en iyi aday, genellikle eski leader'dan en güncel veri değişikliklerine sahip olan replica'dır (herhangi bir veri kaybını minimize etmek için).

3. **Sistemin Yeni Leader'ı Kullanacak Şekilde Yeniden Yapılandırılması:** İstemcilerin artık yazım isteklerini yeni leader'a göndermesi gerekir. Eski leader geri gelirse, hâlâ leader olduğuna inaniyor olabilir ve diğer replika'ların onu geride bıraktığını fark etmemiş olabilir. Sistem, eski leader'ın follower olmasını ve yeni leader'ı tanımasını sağlamalıdır.

### Failover'da Neler Yanlış Gidebilir?

**1. Asenkron Replikasyonda Veri Kaybı:**
Asenkron replikasyon kullanılıyorsa, yeni leader arızalanmadan önce eski leader'dan tüm yazımları almamış olabilir. Eski leader çöktükten sonra yeni bir leader seçilmişse, eski leader'ın replike edilmemiş yazımlarına ne olacak? En yaygın çözüm, eski leader'ın replike edilmemiş yazımlarını **discard** etmektir (atmak). Bu, kesinleşmiş olduğuna inandığınız yazımların aslında kalıcı olmadığı anlamına gelir.

**2. Diğer Sistemlerle Koordinasyon Sorunu:**
Yazımları atmak, özellikle veritabanı içeriğiyle koordine edilmesi gereken veritabanı dışı depolama sistemleri varsa tehlikelidir. GitHub'daki bir olayda: Güncel olmayan bir MySQL follower leader'a yükseltildi. Veritabanı yeni satırlara primary key atamak için otomatik artan bir sayaç kullanıyordu, ancak yeni leader'ın sayacı eski leader'ın gerisinde kaldığından, eski leader tarafından önceden atanmış bazı primary key'leri yeniden kullandı. Bu primary key'ler bir Redis store'unda da kullanıldığından, primary key'lerin yeniden kullanılması MySQL ile Redis arasında tutarsızlığa neden oldu ve bazı özel verilerin yanlış kullanıcılara gösterilmesine yol açtı.

**3. Split Brain:**
Belirli arıza senaryolarında, iki node aynı anda leader olduklarına inanabilir. Bu duruma **split brain** (beyin bölünmesi) denir ve tehlikelidir. Her iki leader da yazımları kabul ederse ve çakışmaları çözmek için bir süreç yoksa veriler muhtemelen kaybolur veya bozulur. Bir güvenlik mekanizması olarak bazı sistemlerde iki leader tespit edilirse bir node'u kapatma mekanizması bulunur. Ancak bu mekanizma dikkatli tasarlanmazsa her iki node da kapanabilir.

**4. Timeout Ayarı:**
Leader ölü ilan edilmeden önce doğru timeout'u belirlemek zordur. Uzun bir timeout, leader arızalandığında kurtarma süresinin uzun olması anlamına gelir. Ancak timeout çok kısa ise gereksiz failover'lar yaşanabilir. Örneğin, geçici bir yük artışı node'un yanıt süresini timeout'un üzerine çıkarabilir veya bir ağ bozulması paketlerin gecikmesine neden olabilir. Sistem zaten yüksek yük veya ağ sorunlarıyla mücadele ediyorsa, gereksiz bir failover durumu daha da kötüleştirebilir.

**Fencing (Çitleme):**
Split brain'e karşı eski leader'ları sınırlandırarak veya kapatarak koruma **fencing** olarak bilinir.

Bu sorunların kolay çözümleri yoktur. Bu nedenle bazı operasyon ekipleri, yazılım otomatik failover'ı desteklese bile failover'ı manuel olarak gerçekleştirmeyi tercih eder.

### Failover'da Hangi Follower Seçilmeli?

Failover'da en önemli şey, **güncel bir follower'ı yeni leader olarak seçmektir.** Senkron veya yarı senkron replikasyon kullanılıyorsa, bu eski leader'ın yazımları onaylamadan önce beklediği follower olacaktır. Asenkron replikasyonda, en yüksek log sequence number'a sahip follower'ı seçebilirsiniz. Bu, failover sırasında kaybedilen veri miktarını minimize eder; birkaç saniyelik yazım kaybı tolere edilebilir olabilir, ancak birkaç günlük geride kalan bir follower'ın seçilmesi felakete yol açabilir.

---
## 8. Replication Log Implementasyonları <a id='8-replication-log-impl'></a>

Leader tabanlı replikasyon perde arkasında nasıl çalışır? Pratikte birkaç farklı replikasyon yöntemi kullanılmaktadır.

### 8.1 Statement-Based Replication (İfade Tabanlı Replikasyon)

En basit durumda, leader yürüttüğü her yazma isteğini (**statement**) kaydeder ve bu statement log'unu follower'larına gönderir. İlişkisel bir veritabanı için bu, her `INSERT`, `UPDATE` veya `DELETE` ifadesinin follower'lara iletildiği anlamına gelir ve her follower, SQL ifadesini sanki bir istemciden almış gibi çözümleyip (parse) yürütür.

**Sorunlar:**

Bu replikasyon yaklaşımı makul görünse de çeşitli şekillerde bozulabilir:

1. **Nondeterministic Fonksiyonlar:** Geçerli tarih ve saati almak için `NOW()` veya rastgele sayı almak için `RAND()` gibi nondeterministik (belirlenimci olmayan) bir fonksiyon çağıran herhangi bir ifade, her replica'da muhtemelen farklı bir değer üretecektir.

2. **Autoincrement ve Sıralı Çalıştırma:** Autoincrement sütunu kullanan ya da veritabanındaki mevcut veriye bağlı olan (örneğin `UPDATE ... WHERE <koşul>`) ifadeler her replica'da tam olarak aynı sırada çalıştırılmalıdır, aksi takdirde farklı bir etkiye sahip olabilirler. Birden fazla eşzamanlı transaction olduğunda bu kısıtlayıcı olabilir.

3. **Side Effect'ler:** Yan etkileri olan ifadeler (trigger'lar, stored procedure'lar, user-defined function'lar gibi) her replica'da farklı yan etkilere yol açabilir.

**Çözüm Yolları:**
Bu sorunların üstesinden gelinebilir. Örneğin, leader nondeterministik fonksiyon çağrılarını log'a kaydedildiğinde sabit bir dönüş değeriyle değiştirebilir, böylece tüm follower'lar aynı değeri alır. Belirleyici ifadeleri sabit bir sırayla çalıştırma fikri, **state machine replication** olarak da bilinir.

Statement-based replikasyon MySQL 5.1 öncesinde kullanılmıştır. Hâlâ zaman zaman kullanılmaktadır, ancak MySQL artık bir ifadede nondeterminizm varsa varsayılan olarak row-based replikasyona (aşağıda açıklanmıştır) geçer.

### 8.2 Write-Ahead Log (WAL) Shipping

B-tree depolama motorlarını crash sonrasında tutarlı bir duruma geri yüklemek için **write-ahead log** (WAL) gereklidir. WAL, indeksleri ve heap'i tutarlı bir duruma geri yüklemek için gerekli tüm bilgileri içerdiğinden, başka bir node üzerinde bir replica oluşturmak için de aynı log'u kullanabiliriz. Leader log'u diske yazmanın yanı sıra ağ üzerinden de follower'larına gönderir. Follower bu log'u işlediğinde, leader'dakiyle aynı dosyaların tam bir kopyasını oluşturur.

Bu yöntem PostgreSQL ve Oracle'da kullanılmaktadır.

**Ana Dezavantaj:**
WAL, verileri çok düşük bir seviyede tanımlar; bir WAL, hangi disk bloklarında hangi byte'ların değiştiğinin ayrıntılarını içerir. Bu, replikasyonu depolama motoruna sıkıca **bağlı (coupled)** hale getirir. Veritabanı bir sürümden diğerine depolama formatını değiştirirse, leader ve follower'larda farklı sürümlerde veritabanı yazılımı çalıştırmak genellikle mümkün değildir.

Replikasyon protokolü follower'ın leader'dan daha yeni bir yazılım sürümü kullanmasına izin verirse, follower'ları yükselterek ve ardından biri yeni leader yaparak yazılımı **sıfır downtime** ile yükseltebilirsiniz. WAL shipping ile bu sürüm uyumsuzluğuna izin verilmediğinden, bu yükseltmeler downtime gerektirir.

### 8.3 Logical (Row-Based) Log Replication

Alternatif olarak, replikasyon log'u ile depolama motoru için farklı log formatları kullanılabilir; bu, replikasyon log'unun depolama motoru iç yapısından **bağımsız (decoupled)** olmasına olanak tanır. Bu tür replikasyon log'una **logical log** denir; depolama motorunun (fiziksel) veri temsilinden ayırt etmek için.

İlişkisel bir veritabanı için logical log, genellikle veritabanı tablolarına **satır granülaritesinde** yazımları açıklayan kayıtların bir dizisidir:

- **Eklenen satır (insert):** Log, tüm sütunların yeni değerlerini içerir.
- **Silinen satır (delete):** Log, silinen satırı benzersiz biçimde tanımlamak için yeterli bilgi içerir. Genellikle bu, primary key'dir; ancak tabloda primary key yoksa, tüm sütunların eski değerlerinin log'a kaydedilmesi gerekir.
- **Güncellenen satır (update):** Log, güncellenen satırı benzersiz biçimde tanımlamak için yeterli bilgi ve tüm sütunların yeni değerlerini (veya en azından değişen sütunların değerlerini) içerir.

Birkaç satırı değiştiren bir transaction, bu tür birkaç log kaydı oluşturur ve ardından transaction'ın commit edildiğini belirten bir kayıt gelir.

MySQL row-based replikasyon kullanacak şekilde yapılandırıldığında, WAL'a ek olarak **binlog** adı verilen ayrı bir logical replikasyon log'u tutar. PostgreSQL, fiziksel WAL'ı satır ekleme/güncelleme/silme olaylarına çözerek (decode ederek) logical replikasyonu uygular.

**Avantajlar:**
- Logical log, depolama motoru iç yapısından bağımsız olduğundan, **backward compatible** şekilde kolaylıkla saklanabilir; bu da leader ve follower'ın farklı sürümlerde veritabanı yazılımı çalıştırmasına olanak tanır. Bu da minimum downtime ile yeni bir sürüme yükseltmeyi mümkün kılar.
- Logical log formatının dış uygulamalar tarafından ayrıştırılması daha kolaydır. Bu, veritabanının içeriğini çevrimdışı analiz için bir data warehouse gibi harici bir sisteme, özel index'ler ve cache'ler oluşturmak için özel bir sisteme göndermek istiyorsanız faydalıdır. Bu tekniğe **change data capture (CDC)** denir.

---
## 9. Replication Lag Sorunları <a id='9-replication-lag'></a>

Node arızalarını tolere etme, replikasyonu istemenin yalnızca bir nedenidir. Diğer nedenler arasında **scalability** (tek bir makinenin kaldırabileceğinden daha fazla isteği işlemek) ve **latency** (replika'ları kullanıcılara coğrafi olarak daha yakın yerleştirmek) sayılabilir.

### Read-Scaling (Okuma Ölçekleme) Mimarisi

Leader tabanlı replikasyon, tüm yazımların tek bir node üzerinden geçmesini gerektirir; ancak salt okunur sorgular herhangi bir replica'ya gidebilir. Büyük çoğunluğu okumalardan oluşan iş yükleri için (genellikle çevrimiçi hizmetlerde böyledir) cazip bir seçenek vardır: çok sayıda follower oluşturun ve okuma isteklerini bu follower'lara dağıtın. Bu, leader'ın üzerindeki yükü azaltır ve okuma isteklerine yakın replica'lar tarafından hizmet verilmesini sağlar.

Bu **read-scaling** mimarisinde, yalnızca daha fazla follower ekleyerek salt okunur isteklere hizmet kapasitesini artırabilirsiniz. Ancak bu yaklaşım gerçekçi olarak yalnızca asenkron replikasyonla işe yarar.

### Eventual Consistency (Nihai Tutarlılık)

Ne yazık ki, **asenkron** bir follower'dan okuyan bir uygulama, follower geride kalmışsa güncel olmayan (outdated) bilgiler görebilir. Bu, veritabanında görünür tutarsızlıklara yol açar: aynı sorguyu aynı anda leader ve bir follower üzerinde çalıştırırsanız, tüm yazımlar follower'a yansıtılmadığından farklı sonuçlar alabilirsiniz. Bu tutarsızlık geçici bir durumdur; veritabanına yazmayı bırakır ve bir süre beklerseniz, follower'lar er ya da geç lider'a yetişecek ve onunla tutarlı hale gelecektir. Bu nedenle bu etkiye **eventual consistency** (nihai tutarlılık) denir.

"Eventually" ifadesi kasıtlı olarak belirsizdir; genel olarak bir replica'nın ne kadar geride kalabileceğine dair bir sınır yoktur. Normal operasyonda, leader'da gerçekleşen bir yazım ile follower'a yansıması arasındaki gecikme — **replication lag** — yalnızca bir saniyenin küçük bir kesri olabilir. Ancak sistem kapasiteye yakın çalışıyorsa veya ağda bir sorun varsa, lag kolaylıkla birkaç saniyeye, hatta dakikalara çıkabilir.

### 9.1 Read-Your-Writes Consistency

Birçok uygulama, kullanıcının bazı veriler göndermesine ve ardından gönderdiklerini görüntülemesine olanak tanır. Yeni veriler gönderildiğinde leader'a gönderilmeli, ancak kullanıcı veriyi görüntülediğinde bir follower'dan okunabilir.

Asenkron replikasyonla bir sorun ortaya çıkar: Kullanıcı yazma yaptıktan kısa süre sonra veriyi görüntülerse, yeni veri henüz replica'ya ulaşmamış olabilir. Kullanıcıya, gönderdiği verilerin kaybolmuş gibi görünür.

Bu durumda **read-after-write consistency** (yazma sonrası okuma tutarlılığı), aynı zamanda **read-your-writes consistency** olarak da bilinen bir garantiye ihtiyaç duyulur. Bu, kullanıcının sayfayı yenilediğinde her zaman kendisinin gönderdiği güncellemeleri göreceğini garanti eder. Diğer kullanıcıların güncellemeleri hakkında hiçbir söz verilmez; ancak kullanıcının kendi girdisinin doğru kaydedildiği konusunda güvence sağlar.

**Implementasyon Teknikleri:**

1. Kullanıcının değiştirmiş olabileceği bir şeyi okurken, leader'dan veya senkron olarak güncellenmiş bir follower'dan okuyun; aksi takdirde asenkron olarak güncellenmiş bir follower'dan okuyun. Bu, neyin değiştirilmiş olabileceğini sorgulamadan bilme yolunuzu gerektirir. Örneğin, bir sosyal ağdaki kullanıcı profil bilgisi normalde yalnızca profilin sahibi tarafından düzenlenebilir. Basit bir kural: kullanıcının kendi profilini her zaman leader'dan okuyun; başka kullanıcıların profillerini ise follower'dan okuyun.

2. Uygulamadaki çoğu şey kullanıcı tarafından potansiyel olarak düzenlenebiliyorsa, çoğu şeyin leader'dan okunması gerekecektir (bu, okuma ölçeklemenin faydasını ortadan kaldırır). Bu durumda, okuma yapılacak yere karar vermek için başka kriterler kullanılabilir. Örneğin, son güncellemenin zamanını takip edebilir ve son güncellemeden itibaren bir dakika boyunca tüm okumaları leader'dan yapabilirsiniz. Ayrıca follower'lardaki replikasyon lag'ini izleyebilir ve leader'dan bir dakikadan fazla geride kalan herhangi bir follower üzerinde sorgu yapılmasını engelleyebilirsiniz.

3. İstemci, en son yazımının timestamp'ini hatırlayabilir ve sistem, o kullanıcı için herhangi bir okumaya hizmet eden replica'nın en azından bu timestamp'e kadar güncellemeleri yansıttığını güvence altına alabilir. Timestamp, log sequence number gibi bir **logical timestamp** veya gerçek sistem saati olabilir.

4. Replica'larınız birden fazla region'a dağıtılmışsa, leader içeren region'a yönlendirilmesi gereken her istek ek karmaşıklık doğurur.

**Cross-Device Read-After-Write Consistency:**
Aynı kullanıcı masaüstü web tarayıcısı ve mobil uygulama gibi birden fazla cihazdan hizmetinize erişiyorsa, ek zorluklar ortaya çıkar: Bir cihazda bilgi girilip başka bir cihazda görüntülendiğinde, kullanıcı girdiği bilgileri görmeli.

### 9.2 Monotonic Reads (Monoton Okumalar)

Asenkron follower'lardan okurken oluşabilecek ikinci anomali türü, kullanıcının zamanın **geriye gittiğini** görmesidir.

Bu, kullanıcı farklı replica'lardan birden fazla okuma yaptığında oluşabilir. Örneğin, kullanıcı 2345 aynı sorguyu önce az gecikmeli bir follower'a, sonra daha fazla gecikmeye sahip bir follower'a gönderir. İlk sorgu, kullanıcı 1234 tarafından yakın zamanda eklenen bir yorumu döndürür, ancak ikinci sorgu hiçbir şey döndürmez; çünkü gecikmiş follower bu yazımı henüz almamıştır. Kullanıcı 2345 açısından zamanın geriye gittiği görünür.

**Monotonic reads**, bu tür anomalinin yaşanmamasını garanti eder. Güçlü tutarlılıktan (strong consistency) zayıf, eventual consistency'den güçlü bir garantidir. Veri okurken eski bir değer görebilirsiniz; monotonic reads yalnızca şunu garantiler: bir kullanıcı sırayla birkaç okuma yaptığında, zamanın geriye gitmediğini (yani daha önce daha yeni veri okuduktan sonra daha eski veri okunamayacağını).

**Monotonic reads'i uygulamanın bir yolu**, her kullanıcının okumalarını her zaman aynı replica'dan yapmasını sağlamaktır. Örneğin, replica, rastgele seçmek yerine kullanıcı ID'sinin hash'ine göre seçilebilir. Ancak bu replica arızalanırsa, kullanıcının sorgularının başka bir replica'ya yönlendirilmesi gerekecektir.

### 9.3 Consistent Prefix Reads (Tutarlı Önek Okumaları)

Replikasyon lag anomalilerinin üçüncü örneği **causality** (nedensellik) ihlaliyle ilgilidir. Şu kısa diyaloğu düşünün:

> **Bay Poons:** Ne kadar ileriye bakabiliyorsunuz, Bayan Cake?  
> **Bayan Cake:** Genellikle yaklaşık 10 saniye kadar, Bay Poons.

Bu iki cümle arasında nedensel bir bağımlılık vardır: Bayan Cake, Bay Poons'un sorusunu duyup yanıtladı. Şimdi üçüncü bir kişinin bu konuşmayı follower'lar aracılığıyla dinlediğini hayal edin. Bayan Cake'in söyledikleri az gecikmeli bir follower'dan geçerken, Bay Poons'un söyledikleri daha uzun bir replikasyon lag'ine sahip bir follower üzerinden geçerse, gözlemci şunları duyar:

> **Bayan Cake:** Genellikle yaklaşık 10 saniye kadar, Bay Poons.  
> **Bay Poons:** Ne kadar ileriye bakabiliyorsunuz, Bayan Cake?

Gözlemciye, Bayan Cake'in Bay Poons sormadan önce soruyu yanıtlıyormuş gibi görünür.

Bu tür anomaliyi önlemek için başka bir güvence gerekir: **consistent prefix reads**. Bu güvence, bir dizi yazım belirli bir sırayla gerçekleşirse, bu yazımları okuyan herkesin bunları aynı sırayla göreceğini söyler.

Bu, özellikle **sharded** (bölümlenmiş) veritabanlarında sorun teşkil eder. Veritabanı yazımları her zaman aynı sırayla uyguluyorsa, okumalar her zaman tutarlı bir önek görür. Ancak pek çok distributed veritabanında farklı shard'lar bağımsız olarak çalışır, dolayısıyla yazımların global bir sırası yoktur.

### Çözümler: Replication Lag için Ne Yapmalı?

Eventual consistent bir sistemle çalışırken, replikasyon lag'i birkaç dakika veya saate çıksa uygulamanın nasıl davranacağını düşünmek önemlidir.

Bir uygulamanın, altta yatan veritabanından daha güçlü bir güvence sağlamasının yolları vardır; örneğin belirli türdeki okumaları leader veya senkron olarak güncellenen follower üzerinde gerçekleştirmek. Ancak bu sorunları uygulama kodunda yönetmek karmaşık ve hata yapmaya açıktır.

Uygulama geliştiricileri için en basit programlama modeli, replica'lar için **linearizability** (doğrusallaştırılabilirlik) gibi güçlü bir tutarlılık garantisi sunan ve **ACID transaction**'ları destekleyen bir veritabanı seçmektir. Bu, replikasyondan kaynaklanan zorlukları büyük ölçüde görmezden gelmenizi ve veritabanını tek bir node'muş gibi ele almanızı sağlar.

2010'ların başında NoSQL hareketi, bu özelliklerin ölçeklenebilirliği sınırladığını ve büyük ölçekli sistemlerin eventual consistency'yi benimsemesi gerektiğini savundu. Ancak o tarihten bu yana bir dizi veritabanı, distributed bir veritabanının hata toleransı, yüksek erişilebilirlik ve ölçeklenebilirlik avantajlarını sunarken güçlü tutarlılık ve transaction desteği sağlamaya başlamıştır. Bu trend, **NewSQL** olarak bilinmektedir.

---
## 10. Multi-Leader Replication <a id='10-multi-leader-replication'></a>

Şimdiye kadar yalnızca tek bir leader kullanan replikasyon mimarilerini ele aldık. Bu yaygın bir yaklaşım olsa da ilginç alternatifler de mevcuttur.

Single-leader replikasyonun önemli bir dezavantajı vardır: tüm yazımların tek bir node üzerinden geçmesi zorunludur. Herhangi bir nedenle leader'a bağlanamazsanız — örneğin siz ile leader arasında bir ağ kesintisi varsa — veritabanına yazamazsınız.

Single-leader replikasyon modelinin doğal bir uzantısı, birden fazla node'un yazımları kabul etmesine izin vermektir. Replikasyon aynı şekilde gerçekleşmeye devam eder: yazım işleyen her node, bu veri değişikliğini tüm diğer node'lara iletmek zorundadır. Bu yapılandırmaya **multi-leader** (çok liderli) yapılandırma (aynı zamanda **active/active** veya **bidirectional** replikasyon olarak da bilinir) denir. Bu kurulumda her leader aynı zamanda diğer leader'ların follower'ı olarak da işlev görür.

Single-leader replikasyonda olduğu gibi, senkron mu yoksa asenkron mu olacağı seçimi burada da geçerlidir. İki leader olan bir sistemde, A'ya senkron yazım B'ye replike edilirse ve iki node arasındaki ağ kesintiye uğrarsa, bağlantı yeniden kurulana kadar A'ya yazamazsınız. Bu nedenle, senkron çok liderli replikasyon aslında tek liderli replikasyona eşdeğerdir. Dolayısıyla bu bölümün geri kalanı **asenkron multi-leader replikasyona** odaklanacaktır.

### 10.1 Coğrafi Olarak Dağıtılmış Operasyon (Geographically Distributed Operation)

Multi-leader replikasyonu tek bir region içinde kullanmak nadiren mantıklıdır; faydaları nadiren eklenen karmaşıklığı haklı kılar. Ancak bazı durumlarda bu yapılandırma makul olabilir.

Birden fazla region'da replica'ları olan bir veritabanınız olduğunu düşünün (belki tüm bir region'ın arızasına dayanmak veya kullanıcılarınıza yakınlık için). Bu, **geo-distributed** (coğrafi olarak dağıtılmış) veya **geo-replicated** (coğrafi olarak çoğaltılmış) bir kurulum olarak bilinir. Single-leader replikasyonda, leader'ın tek bir region'da bulunması ve tüm yazımların o region üzerinden geçmesi gerekir.

Multi-leader yapılandırmasında, her region'da bir leader olabilir. Her region içinde normal leader-follower replikasyon kullanılır; region'lar arasında ise her region'ın leader'ı değişikliklerini diğer region'lardaki leader'lara replike eder.

**Single-Leader ile Multi-Leader Karşılaştırması:**

| Boyut | Single-Leader | Multi-Leader |
|-------|--------------|-------------|
| **Performans** | Her yazım leader region'una internet üzerinden gitmeli; bu önemli gecikme ekler | Her yazım yerel region'da işlenip asenkron replike edilebilir; gecikme kullanıcılardan gizlenir |
| **Regional Arıza Toleransı** | Leader region'u kullanılamaz hale gelirse follower bir başka region'da leader olarak yükseltilir | Her region diğerlerinden bağımsız çalışmaya devam edebilir |
| **Ağ Sorunları Toleransı** | Region'lar arası geçici ağ sorunlarına karşı çok hassas | Region'lar arası geçici ağ kesintilerini daha iyi tolere eder |
| **Tutarlılık** | Serializable transaction gibi güçlü tutarlılık garantileri sunabilir | Tutarlılık çok daha zayıf; banka hesabı sıfırın altına inebilir, benzersiz kullanıcı adı garantisi verilemez |

Multi-leader replikasyon, single-leader replikasyona göre daha az yaygındır, ancak MySQL, Oracle, SQL Server ve YugabyteDB dahil birçok veritabanı tarafından desteklenmektedir.

Multi-leader replikasyon, birçok veritabanında sonradan eklenen bir özellik olduğundan, diğer veritabanı özellikleriyle sürpriz etkileşimler görülebilir. Örneğin, autoincrement key'ler, trigger'lar ve bütünlük kısıtlamaları sorunlu olabilir. Bu nedenle multi-leader replikasyon, mümkünse kaçınılması gereken tehlikeli bir alan olarak değerlendirilir.

### 10.2 Multi-Leader Replication Topologies

**Replication topology**, yazımların bir node'dan diğerine nasıl yayıldığını açıklayan iletişim yollarını tanımlar. İki lider varsa, yalnızca bir topoloji mantıklıdır: leader 1 tüm yazımlarını leader 2'ye göndermeli ve bunun tersi de geçerli. İkiden fazla leader olduğunda çeşitli topolojiler mümkündür:

- **All-to-All (Hepsi-Her-Şeye):** En genel topoloji. Her leader, yazımlarını diğer tüm leader'lara gönderir.
- **Circular (Döngüsel):** Her node, bir node'dan yazımları alır ve bu yazımları (kendi yazımlarıyla birlikte) başka bir node'a iletir. MySQL varsayılan olarak bunu kullanır.
- **Star (Yıldız):** Belirlenen bir kök node, yazımları diğer tüm node'lara iletir. Yıldız topolojisi bir ağaca genelleştirilebilir.

**Sonsuz Döngü Önlemi:**
Döngüsel ve yıldız topolojilerinde, bir yazımın tüm replika'lara ulaşmadan önce birkaç node'dan geçmesi gerekebilir. Bu nedenle node'ların diğer node'lardan aldıkları veri değişikliklerini iletmesi gerekir. Sonsuz replikasyon döngülerini önlemek için her node benzersiz bir tanımlayıcıya sahiptir ve replikasyon log'unda her yazım geçtiği tüm node'ların tanımlayıcılarıyla etiketlenir. Bir node kendi tanımlayıcısıyla etiketlenmiş bir veri değişikliği alırsa, bu veri değişikliği görmezden gelir.

**Farklı Topolojilerin Sorunları:**
Döngüsel ve yıldız topolojilerindeki bir sorun, tek bir node arızalandığında diğer node'lar arasındaki replikasyon mesajlarının akışını kesintiye uğratabilmesidir; node düzelene kadar iletişim kuramaz hale gelebilirler. Daha yoğun bağlantılı bir topoloji (all-to-all gibi) daha iyi hata toleransına sahiptir; çünkü mesajların tek bir arıza noktasından kaçınarak farklı yolları izlemesine olanak tanır.

Ancak all-to-all topolojilerinde de sorunlar olabilir. Özellikle bazı ağ bağlantıları diğerlerinden daha hızlı olabilir (örneğin ağ tıkanıklığı nedeniyle); bu da bazı replikasyon mesajlarının diğerlerini "geçmesine" yol açabilir. Örneğin: client A bir tabloya bir satır ekler ve client B bu satırı günceller. Leader 2 önce güncellemeyi, ardından eklemeyi alabilir. Güncelleme, veritabanında var olmayan bir satıra güncelleme olarak leader 2'nin bakış açısından görünür.

---
## 11. Sync Engines ve Local-First Software <a id='11-sync-engines'></a>

Multi-leader replikasyon aynı zamanda internetten bağlantısı kesildiğinde çalışmaya devam etmesi gereken uygulamalar için de uygundur. Örneğin, mobil telefonunuzdaki, dizüstü bilgisayarınızdaki ve diğer cihazlarınızdaki takvim uygulamalarını düşünün. Cihazınızın şu anda internet bağlantısı olsun ya da olmasın, toplantılarınızı görebilmeniz (okuma isteği) ve yeni toplantılar girebilmeniz (yazma isteği) gerekir. Çevrimdışıyken herhangi bir değişiklik yaparsanız, cihaz bir sonraki çevrimiçi olduğunda bu değişikliklerin bir sunucu ve diğer cihazlarınızla senkronize edilmesi gerekir.

Bu durumda, her cihazda yazma isteklerini kabul eden bir leader olarak işlev gören yerel bir veritabanı replica'sı bulunur. Replikasyon gecikmesi (lag), internet erişiminize bağlı olarak saatler hatta günler olabilir.

Mimari açıdan bu kurulum, region'lar arası multi-leader replikasyona oldukça benzerdir; ancak çok daha uç bir noktadadır. Her cihaz bir "region" ve aralarındaki ağ bağlantısı son derece güvenilmezdir.

### Real-Time Collaboration, Offline-First ve Local-First Uygulamalar

Birçok modern web uygulaması **real-time collaboration** (gerçek zamanlı işbirliği) özellikleri sunar: metin belgeleri ve e-tablolar için Google Docs ve Sheets, grafikler için Figma, proje yönetimi için Linear. Bu uygulamaları bu kadar duyarlı yapan şey, kullanıcı girdisinin sunucuya bir ağ gidiş-dönüşü beklemeksizin anında kullanıcı arayüzüne yansıtılması ve bir kullanıcının düzenlemelerinin işbirliği yaptığı kişilere düşük gecikmeyle gösterilmesidir.

Bu bir multi-leader mimarisiyle sonuçlanır: paylaşılan dosyayı açmış her web tarayıcı sekmesi bir replica'dır ve dosyada yaptığınız güncellemeler, aynı dosyayı açmış diğer kullanıcıların cihazlarına asenkron olarak replike edilir.

Hem çevrimdışı düzenleme hem de gerçek zamanlı işbirliği benzer bir replikasyon altyapısı gerektirir. Uygulama; kullanıcının dosyada yaptığı değişiklikleri yakalayıp ya hemen işbirliği yapanlara göndermeli (çevrimiçiyse) ya da daha sonra göndermek üzere yerel olarak depolamalıdır (çevrimdışıysa). Ayrıca uygulama; işbirliği yapanlardan gelen değişiklikleri almalı, dosyanın kullanıcının yerel kopyasıyla birleştirmeli ve en son sürümü yansıtmak üzere UI'ı güncellemelidir.

### Sync Engine Nedir?

Bu süreci destekleyen yazılım kütüphanelerine **sync engine** denir. Bu fikir uzun süredir vardır, ancak terim yakın zamanda ilgi kazanmıştır.

- **Offline-first:** Bir kullanıcının çevrimdışıyken dosyayı düzenlemeye devam etmesine olanak tanıyan uygulama.
- **Local-first software:** Yalnızca offline-first olmakla kalmayıp, yazılımı geliştiren şirketin tüm çevrimiçi hizmetlerini kapatması durumunda bile çalışmaya devam edecek şekilde tasarlanmış işbirlikçi uygulamalara verilen ad. Bu, birden fazla hizmet sağlayıcının bulunduğu açık bir standart senkronizasyon protokolü kullanan bir sync engine ile sağlanabilir. Örneğin, Git bir local-first işbirliği sistemidir (gerçek zamanlı işbirliğini desteklemese de); GitHub, GitLab veya başka herhangi bir repository hosting hizmeti üzerinden senkronize edilebilir.

### Sync Engine'nin Avantajları

Günümüzde web uygulamalarını geliştirmenin baskın yolu, istemcide çok az kalıcı durum tutmak ve yeni bir veri parçasının görüntülenmesi veya bazı verilerin güncellenmesi gerektiğinde sunucuya istekler göndermektir. Buna karşın, bir sync engine kullanıldığında, istemcide kalıcı durum bulunur ve sunucuyla iletişim bir arka plan sürecine taşınır.

1. **Yüksek Hız:** Verinin yerel olarak bulunması, bir hizmet çağrısının sonucunu beklemeye kıyasla UI'ın çok daha hızlı yanıt vermesini sağlar. Bazı uygulamalar, 60 Hz yenileme hızına sahip bir ekranda 16 ms içinde render edilmesi anlamına gelen grafik sisteminin **sonraki frame'inde** kullanıcı girdisine yanıt vermeyi hedefler.

2. **Offline Çalışma:** Kullanıcıların çevrimdışı çalışmaya devam edebilmesi, özellikle kesintili bağlantılı mobil cihazlarda değerlidir. Bir sync engine ile uygulamanın ayrı bir offline modu yoktur: çevrimdışı olmak, çok büyük bir ağ gecikmesiyle aynıdır.

3. **Daha Basit Programlama Modeli:** Sync engine, açık hizmet çağrılarına kıyasla frontend uygulamalar için programlama modelini basitleştirir. Her hizmet çağrısı hata işleme gerektirir; bir sync engine, uygulamanın yerel veriler üzerinde okuma ve yazma yapmasına olanak tanır; bu işlemler neredeyse hiç başarısız olmaz.

4. **Reactive Programming ile Real-Time Güncellemeler:** Diğer kullanıcıların düzenlemelerini gerçek zamanlı olarak görüntülemek için bu düzenlemelere ilişkin bildirimler alınmalı ve UI verimli şekilde güncellenmelidir. **Reactive programming** modeliyle birleştirilmiş bir sync engine, bunu uygulamanın iyi bir yoludur.

### Sync Engine'nin Sınırları

Sync engine'ler en iyi, kullanıcının ihtiyaç duyabileceği tüm verilerin önceden indirilip kalıcı olarak istemcide depolandığı durumlarda çalışır. Bu, verinin çevrimdışı erişim için mevcut olduğu anlamına gelir, ancak aynı zamanda sync engine'lerin kullanıcının çok büyük miktarda veriye erişimi olduğunda uygun olmadığını da gösterir. Örneğin, kullanıcının oluşturduğu tüm dosyaları indirmek makul olabilir, ancak bir e-ticaret web sitesinin tüm kataloğunu indirmek muhtemelen mantıklı değildir.

### Mevcut Sync Engine'ler

- **Özel backend:** Google Firestore, Realm, Ditto
- **Açık kaynak:** PouchDB/CouchDB, Automerge, Yjs
- **Oyunlarda eşdeğeri:** Multiplayer video oyunları, kullanıcının yerel eylemlerine anında yanıt verme ve bunları ağ üzerinden asenkron olarak alınan diğer oyuncuların eylemleriyle uzlaştırma ihtiyacına benzer bir ihtiyaç duyar. Oyun geliştirme jargonunda, bir sync engine'nin eşdeğerine **netcode** denir.

---
## 12. Conflicting Write'larla Başa Çıkma (Dealing with Conflicting Writes) <a id='12-conflict-resolution'></a>

Multi-leader replikasyonun en büyük sorunu, farklı leader'larda eşzamanlı yazımların çözümlenmesi gereken **conflict**'lere (çakışmalara) yol açabilmesidir.

Örneğin, iki kullanıcının eşzamanlı olarak bir wiki sayfasını düzenlediğini düşünün: Kullanıcı 1, sayfanın başlığını A'dan B'ye değiştirir ve kullanıcı 2, başlığı A'dan C'ye değiştirir. Her kullanıcının değişikliği kendi yerel leader'ında başarıyla uygulanır. Ancak değişiklikler asenkron olarak replike edildiğinde bir çakışma tespit edilir. Bu sorun, single-leader bir veritabanında yaşanmaz.

### 12.1 Conflict Avoidance (Çakışmadan Kaçınma)

Çakışmalarla başa çıkmanın bir stratejisi, çakışmaların baştan oluşmasını engellemektir. Örneğin, uygulama belirli bir kayıt için tüm yazımların aynı leader'dan geçmesini sağlayabilirse, çakışmalar yaşanmaz; veritabanı genel olarak multi-leader olsa bile. Bu yaklaşım, çevrimdışı güncellenen bir sync engine istemcisi için mümkün değildir, ancak bazen coğrafi olarak dağıtılmış sunucu sistemlerinde uygulanabilir.

Örneğin, kullanıcının yalnızca kendi verisini düzenleyebildiği bir uygulamada, belirli bir kullanıcıdan gelen isteklerin her zaman aynı region'a yönlendirilmesini ve okuma ve yazma için o region'daki leader'ın kullanılmasını sağlayabilirsiniz.

Çakışmadan kaçınmanın başka bir örneği olarak, yeni kayıtlar ekleyip benzersiz ID'ler üretmek istediğinizde, iki lideriniz varsa birinin yalnızca tek sayılar üretirken diğerinin çift sayılar üretmesini sağlayabilirsiniz.

Ancak, belirlenen leader'ı değiştirmeye izin verirseniz (bir region kullanılamaz olduğu ve trafiği başka bir region'a yeniden yönlendirmeniz gerektiği için) çakışmadan kaçınma bozulur.

### 12.2 Last Write Wins — LWW (Son Yazım Kazanır)

Çakışmalar önlenemiyorsa, bunları çözmenin en basit yolu her yazıma bir timestamp eklemek ve her zaman en son (en büyük) timestamp'e sahip değeri kullanmaktır. Buna **Last Write Wins** (LWW) — son yazım kazanır — denir.

Terim yanıltıcıdır, çünkü iki yazım eşzamanlı olduğunda hangisinin en son olduğu tanımsızdır; dolayısıyla eşzamanlı yazımların timestamp sırası esasen rastgeledir.

LWW'in gerçek anlamı şudur: aynı kayıt farklı leader'larda eşzamanlı olarak yazıldığında, yazımlardan biri rastgele kazanan seçilir ve diğer yazımlar sessizce atılır; her ne kadar kendi leader'ları tarafından başarıyla işlenmiş olsalar da.

Bu, tüm replica'ların nihayetinde tutarlı bir duruma gelmesi hedefini gerçekleştirir, ancak **veri kaybı** pahasına.

**LWW'nin Timestamp Sorunu:** LWW ile gerçek zamanlı saat (örneğin Unix timestamp) timestamp olarak kullanılıyorsa, sistem **saat senkronizasyonuna** karşı son derece hassas hale gelir. Eğer bir node'un saati diğerlerinden ilerideyse ve bu node tarafından yazılan bir değerin üzerine yazmaya çalışırsanız, yazımınız daha düşük timestamp'e sahip olduğundan görmezden gelinebilir. Bu sorun, daha sonra ele alacağımız bir **logical clock** kullanılarak çözülebilir.

### 12.3 Manual Conflict Resolution (Manuel Çakışma Çözümü)

Bazı yazımların rastgele atılması istenmiyorsa, bir sonraki seçenek çakışmayı **manuel olarak** çözmektir. Bu, Git ve diğer versiyon kontrol sistemlerindeki manuel çakışma çözümüne benzer: iki branch'teki commit'ler aynı dosyanın aynı satırlarını düzenlerse ve bu branch'leri birleştirmeye çalışırsanız, birleştirme tamamlanmadan önce çözülmesi gereken bir merge conflict alırsınız.

Bir veritabanında, bir çakışmanın tüm replikasyon sürecini bir insan çözene kadar durdurması pratik olmaz. Bunun yerine, veritabanları genellikle belirli bir kayıt için eşzamanlı olarak yazılmış tüm değerleri saklar; bazen bunlara **siblings** (kardeşler) denir. O kaydı bir sonraki sorgulamanızda, veritabanı yalnızca en son değeri değil, *tüm* bu değerleri döndürür. Ardından bu değerleri istediğiniz şekilde çözebilirsiniz.

**Manuel Çözümün Dezavantajları:**
- Veritabanı API'si değişir; wiki sayfasının başlığı artık yalnızca bir string olmak yerine genellikle bir eleman içeren, ancak bazen çakışma varsa birden fazla eleman içerebilen bir string kümesi haline gelir.
- Sibling'leri manuel olarak birleştirmeyi kullanıcıdan istemek, hem uygulama geliştiricisi hem de kullanıcı için çok fazla iş demektir.
- Sibling'leri otomatik olarak birleştirmek, dikkatli yapılmazsa sürpriz davranışlara yol açabilir. Amazon'un alışveriş sepeti bunu yapıyordu; sibling'leri birleştirmek için set union (küme birleşimi) alıyordu. Bu, bir müşteri kendi sepetindeki bir ürünü silip başka bir sibling hâlâ o eski ürünü içeriyorsa, silinen ürünün beklenmedik biçimde sepette yeniden görüneceği anlamına geliyordu.
- Birden fazla node çakışmayı gözlemlerse ve bunu eşzamanlı olarak çözerlerse, çakışma çözüm süreci kendisi yeni bir çakışma yaratabilir.

### 12.4 Automatic Conflict Resolution (Otomatik Çakışma Çözümü)

Birçok uygulama için çakışmaları yönetmenin en iyi yolu, eşzamanlı yazımları tutarlı bir durumda otomatik olarak birleştiren bir algoritma kullanmaktır. Otomatik çakışma çözümü, tüm replica'ların **converge** etmesini (birleşmesini) sağlar; yani aynı yazım kümesini işlemiş tüm replica'lar yazımların geliş sırasına bakılmaksızın aynı duruma sahip olur. Eventual consistency ile birleşim garantisinin bir araya getirilmesi **strong eventual consistency** (güçlü nihai tutarlılık) olarak bilinir.

Farklı veri türleri için daha gelişmiş birleştirme algoritmaları geliştirilmiştir:

- **Metin (text):** Hangi karakterlerin eklenip silindiğini tespit edebiliriz. Birleştirilen sonuç, sibling'lerden herhangi birinde yapılan tüm ekleme ve silmeleri korur.
- **Koleksiyon (liste/küme):** Metin gibi ekleme ve silmeleri takip ederek birleştirilebilir.
- **Sayaç (counter):** Her sibling üzerinde kaç artış ve azaltma gerçekleştiğini söyleyebilir ve bunları doğru biçimde toplayabiliriz.
- **Key-Value mapping:** Aynı key altındaki güncellemeler, diğer çakışma çözüm algoritmalarından biriyle birleştirilebilir.

### 12.5 CRDTs ve Operational Transformation

Otomatik çakışma çözümünü uygulamak için iki algoritma ailesi yaygın olarak kullanılır:

**CRDT (Conflict-Free Replicated Data Type):**
**Operational Transformation (OT):**

Farklı tasarım felsefelerine ve performans özelliklerine sahip olmalarına karşın her ikisi de yukarıda bahsedilen tüm veri türleri için otomatik birleştirme yapabilir.

Örnek: "ice" kelimesi ile başlayan iki replica ele alalım. Bir replica başa `n` ekleyerek "nice" yapar; diğeri ise eşzamanlı olarak sona `!` ekleyerek "ice!" yapar. Birleştirilen sonuç "nice!" olmalıdır.

**OT** indeks tabanlı çalışır: `n`, indeks 0'a eklenir; `!`, indeks 3'e eklenir. Replica'lar işlemlerini değiş tokuş ettikten sonra, `!`'in indeksi daha önceki bir indekse eklenen `n`'yi hesaba katmak için dönüştürülür (4'e güncellenir).

**CRDT** ise indeks yerine her karaktere benzersiz ve değişmez bir ID verir. `!` eklendiğinde, bunun ekleneceği mevcut karakterin ID'sini içeren bir işlem üretilir; bu, eşzamanlı ekleme durumlarında dönüştürme yapmaya gerek bırakmaz.

OT en çok Google Docs gibi metin üzerinde gerçek zamanlı işbirlikçi düzenleme için kullanılırken, CRDT'ler Redis Enterprise, Riak ve Azure Cosmos DB gibi distributed veritabanlarında kullanılmaktadır.

### 12.6 Çakışma Türleri

Bazı çakışma türleri açıktır: iki yazım aynı kaydın aynı alanını eşzamanlı olarak değiştirirse çakışma vardır.

Diğer çakışma türleri daha ince olabilir. Örneğin bir toplantı odası rezervasyon sistemini düşünün. Sistem, hangi odanın hangi zaman diliminde kim tarafından rezerve edildiğini takip eder. Uygulama, her odanın bir seferde yalnızca bir grup tarafından rezerve edilebildiğini güvence altına almalıdır. Aynı oda için aynı anda iki rezervasyon oluşturulursa bir çakışma meydana gelebilir; kullanıcı rezervasyon yapmadan önce müsaitliği kontrol etse bile, rezervasyonlar her ikisi de kaydı eklemeden önce odanın müsait göründüğünü görecek kadar yakın zamanda yapılırsa bir çakışma ortaya çıkabilir.

---
## 13. Leaderless Replication <a id='13-leaderless-replication'></a>

Şimdiye kadar tartışılan replikasyon yaklaşımları — single-leader ve multi-leader — bir istemcinin yazma isteğini tek bir node'a (leader) gönderdiği ve veritabanı sisteminin bu yazımı diğer replika'lara kopyalamayı üstlendiği fikrine dayanır. Bir leader, yazımların işleneceği sırayı belirler ve follower'lar leader'ın yazımlarını aynı sırayla uygular.

Bazı veri depolama sistemleri farklı bir yaklaşım benimser: **leader kavramını tamamen terk ederek** herhangi bir replica'nın doğrudan istemcilerden yazımları kabul etmesine izin verir. En eski replikalı veri sistemlerinden bazıları leaderless'tı, ancak bu fikir ilişkisel veritabanlarının egemenliği döneminde büyük ölçüde unutuldu.

Amazon 2007'de kendi bünyesindeki **Dynamo** sistemi için bunu kullandıktan sonra leaderless replikasyon yeniden moda bir veritabanı mimarisi haline geldi. Riak, Cassandra ve ScyllaDB, Dynamo'dan ilham alan leaderless replikasyon modeline sahip açık kaynaklı veri depolarıdır; bu tür veritabanları **Dynamo-style** olarak da bilinir.

> **Not:** Orijinal Dynamo sistemi Amazon dışında hiç yayımlanmadı. Benzer adlı DynamoDB ise tamamen farklı bir mimariye sahiptir: Multi-Paxos consensus algoritmasına dayalı single-leader replikasyon kullanır.

Bazı leaderless implementasyonlarda istemci yazımlarını doğrudan birden fazla replica'ya gönderir; diğerlerinde ise bir **coordinator node** bunu istemci adına yapar. Ancak bir leader veritabanının aksine, coordinator yazımların belirli bir sırasını uygulamaz.

### 13.1 Bir Node Kapalıyken Veritabanına Yazma

Üç replica'nız olduğunu ve şu an biri kullanılamaz olduğunu hayal edin — belki sistem güncellemesini kurmak için yeniden başlatılıyor. Single-leader yapılandırmasında, yazımları işlemeye devam etmek istiyorsanız failover yapmanız gerekebilir.

Leaderless yapılandırmada ise failover diye bir şey yoktur; tüm replica'lar eşit olduğundan leader yoktur. İstemci, yazımı tüm üç replica'ya paralel olarak gönderir. İki kullanılabilir replica yazımı kabul eder, ancak kullanılamaz olan replica onu kaçırır. İki replika'dan onay alındıktan sonra yazımı başarılı kabul ederiz.

Şimdi kullanılamaz olan node çevrimiçi gelirse, o node olmadığı sürede gerçekleşen yazımlar eksik olacaktır. Bu node'dan okursanız **stale** (bayat) değerler alabilirsiniz.

Bu sorunu çözmek için istemci veritabanından okurken isteğini yalnızca bir replica'ya değil, **paralel olarak birkaç node'a gönderir**. İstemci farklı replica'lardan farklı yanıtlar alabilir; versiyon numarası sayesinde en güncel yanıtı belirler ve yalnızca onu kullanır.

### 13.2 Kaçırılan Yazımları Telafi Etme

Replikasyon sistemi, nihayetinde tüm verilerin her replica'ya kopyalanmasını sağlamalıdır. Kullanılamaz olan node tekrar çevrimiçi geldiğinde, kaçırdığı yazımları nasıl tazeler? Dynamo-style veri depolarında birkaç mekanizma kullanılır:

**Read Repair (Okuma Onarımı):**
İstemci birden fazla node'dan paralel olarak okuma yaptığında, bayat yanıtları tespit edebilir. Örneğin, kullanıcı 2345, replica 3'ten versiyon 6 değeri ve replica 1 ve 2'den versiyon 7 değeri alır. İstemci, replica 3'ün bayat bir değere sahip olduğunu görür ve daha yeni değeri o replica'ya geri yazar. Bu yaklaşım, sık okunan değerler için iyi çalışır.

**Hinted Handoff (İpuçlu Devir Teslim):**
Bir replica kullanılamıyorsa, başka bir replica yazımları onun adına **hint** (ipucu) biçiminde saklayabilir. Yazımları alması gereken replica geri döndüğünde, hint'leri saklayan replica bunları kurtarılan replica'ya gönderir ve ardından hint'leri siler. Bu **handoff** (devir teslim) süreci, hiç okunmayan ve dolayısıyla read repair tarafından ele alınmayan değerler için bile replica'ların güncellenmesine yardımcı olur.

**Anti-Entropy (Entropi Karşıtı Süreç):**
Ayrıca bir arka plan süreci, replica'lar arasındaki veri farklılıklarını periyodik olarak arar ve eksik verileri bir replica'dan diğerine kopyalar. Leader tabanlı replikasyondaki replikasyon log'unun aksine, bu **anti-entropy process** (entropi karşıtı süreç) yazımları belirli bir sırayla kopyalamaz ve verinin kopyalanmasından önce önemli bir gecikme yaşanabilir.

---
## 14. Quorum ile Okuma/Yazma <a id='14-quorum'></a>

### Quorum Koşulu

`n` replica varsa, her yazımın başarılı sayılabilmesi için `w` node tarafından onaylanması ve her okuma için en az `r` node'un sorgulanması gerekir.

**`w + r > n`** koşulu sağlandığında, okunan değerin güncel olmasını bekleriz; çünkü okuduğumuz `r` node'lardan en az biri en güncel olmalıdır.

```
Örnek: n = 3, w = 2, r = 2

     YAZIM                    OKUMA
  [Node 1] ✓             [Node 1] ✓
  [Node 2] ✓             [Node 2] ✓
  [Node 3] ✗ (offline)   [Node 3] ✗ (offline)

w = 2 ✓ (2 onay alındı)  r = 2 ✓ (2 yanıt alındı)
w + r = 4 > n = 3  → overlap garanti edildi!
```

Bu `r` ve `w` değerlerine uyan okuma ve yazmalara **quorum** okumaları ve yazmaları denir. `r` ve `w`'yu, okumanın veya yazmanın geçerli sayılması için gereken minimum oy sayısı olarak düşünebilirsiniz.

### Quorum Parametreleri Yapılandırılabilir

Dynamo-style veritabanlarında `n`, `w` ve `r` parametreleri tipik olarak yapılandırılabilir. Yaygın bir seçim, `n`'yi tek sayı yapmak (genellikle 3 veya 5) ve `w = r = (n + 1) / 2` (yukarı yuvarlanmış) olarak ayarlamaktır. Ancak sayıları uygun gördüğünüz şekilde değiştirebilirsiniz. Örneğin, az yazımlı ve çok okuyuculu bir iş yükünde `w = n` ve `r = 1` ayarlamak faydalı olabilir. Bu, okumaları hızlandırır ancak yalnızca bir node arızası tüm yazımların başarısız olmasına neden olur.

### Quorum Koşulunun Sağlayabileceği Tolerans

- `w < n` ise bir node kullanılamıyorsa hâlâ yazımları işleyebiliriz.
- `r < n` ise bir node kullanılamıyorsa hâlâ okumaları işleyebiliriz.
- `n = 3, w = 2, r = 2` ile bir kullanılamayan node'u tolere edebiliriz.
- `n = 5, w = 3, r = 3` ile iki kullanılamayan node'u tolere edebiliriz.

Normalde okumalar ve yazımlar her zaman `n` replica'ya paralel olarak gönderilir. `w` ve `r` parametreleri, kaç node'u beklediğimizi belirler; yani `n` node'un kaçının başarıyı raporlaması gerektiğini.

### Quorum Tutarlılığının Sınırları

`w + r > n` ile her okumanın yazılmış en son değeri döndüreceğini beklersiniz. Ancak pratikte bu çok daha karmaşıktır. Beklenen davranışın bozulabileceği bazı senaryolar:

1. Yeni bir değer taşıyan bir node arızalanırsa ve verisi eski değer taşıyan bir replica'dan geri yüklenirse, yeni değeri depolayan replica sayısı `w`'nun altına düşebilir; bu, quorum koşulunu bozar.

2. Yeniden dengeleme (rebalancing) devam ederken node'ların belirli bir değer için `n` replica'yı hangi node'ların tutması gerektiğine dair farklı görüşleri olabilir. Bu, okuma ve yazma quorum'larının artık örtüşmemesiyle sonuçlanabilir.

3. Bir okuma, eşzamanlı bir yazım işlemiyle çakışırsa, yeni değeri görebilir de görmeyebilir de. Özellikle bir okumanın yeni değeri görmesi ve ardından bir sonraki okumanın eski değeri görmesi mümkündür.

4. Bir yazım bazı replica'larda başarılı olurken diğerlerinde başarısız olursa (örneğin bazı node'lardaki diskler dolduğundan) ve genel olarak `w` replica'dan azında başarılı olursa, başarısız olan replica'larda geri alınmaz. Bu, bir yazım başarısız olarak raporlanmışsa sonraki okumaların o yazımın değerini döndürüp döndürmeyeceği konusunda belirsizlik yaratır.

5. Veritabanı yeni olan yazımı belirlemek için gerçek zamanlı saat timestamp'leri kullanıyorsa (Cassandra ve ScyllaDB gibi), daha hızlı saate sahip başka bir node aynı key'e yazmışsa yazımlar sessizce atılabilir.

6. İki yazım eşzamanlı gerçekleşirse, bir replica'da önce biri işlenirken diğer bir replica'da önce öbürü işlenebilir. Bu, multi-leader replikasyondaki çakışmalara benzer bir çakışmaya yol açar.

Bu nedenle, quorum'lar okumanın yazılmış en son değeri döndüreceğini garanti ediyor gibi görünse de pratikte bu kadar basit değildir. Dynamo-style veritabanları genellikle eventual consistency'yi tolere edebilen kullanım senaryoları için optimize edilmiştir.

### Staleness Monitörleme

Operasyonel açıdan, veritabanlarınızın güncel sonuçlar döndürip döndürmediğini izlemek önemlidir. Uygulamanız stale okumaları tolere edebilse de replikasyonun sağlığından haberdar olmanız gerekir. Önemli ölçüde geride kalıyorsa, nedeni araştırabilmeniz için bir uyarı vermelisiniz.

Leader tabanlı replikasyonda, veritabanı genellikle bir izleme sistemine besleyebileceğiniz replikasyon lag metrikleri sunar. Bu mümkündür çünkü yazımlar hem leader'a hem de follower'lara aynı sırayla uygulanır ve her node, replikasyon log'unda bir konuma sahiptir. Bir follower'ın mevcut konumunu leader'ın mevcut konumundan çıkararak replikasyon lag miktarını ölçebilirsiniz.

Ancak leaderless replikasyona sahip sistemlerde, yazımların uygulandığı sabit bir sıra yoktur; bu da izlemeyi daha zor hale getirir. Eventual consistency kasıtlı olarak belirsiz bir garantidir, ancak operasyon açısından "eventually"yi sayısal olarak ifade edebilmek önemlidir.

### Single-Leader vs Leaderless Performans

**Single-Leader'ın Performans Sorunları:**
- Leader okuma kapasitesi sınırlıdır (asenkron güncellenen replica'lara okuma dağıtmak stale değerler döndürebilir)
- Leader arızalanırsa failover tamamlanana kadar isteklere hizmet edilemez
- Sistem leader üzerindeki performans sorunlarına son derece hassastır

**Leaderless'ın Avantajları:**
Leaderless mimarisinin büyük avantajı, bu tür sorunlara karşı daha dirençli olmasıdır. Failover olmadığından ve istekler zaten paralel olarak birden fazla replica'ya gönderildiğinden, bir replica'nın yavaş veya kullanılamaz hale gelmesi yanıt süreleri üzerinde çok az etki bırakır; istemci basitçe daha hızlı yanıt veren diğer replica'ların yanıtlarını kullanır. En hızlı yanıtları kullanmak **request hedging** olarak adlandırılır ve kuyruk gecikmesini önemli ölçüde azaltabilir.

Leaderless sistemin direncinin özünde, normal durum ile arıza durumu arasında bir ayrım yapmadığı gerçeği yatar. Bu özellikle, bir node'un tamamen çökmemiş ancak istekleri işlemede olağandışı biçimde yavaş kaldığı **gray failure** (gri arıza) durumlarında faydalıdır.

**Leaderless'ın Performans Sorunları:**
- Replikalar arasındaki kullanılamaz durumlar için hinted handoff gerektirir; bu da sisteme ek yük bindirir
- Daha fazla replica, daha büyük quorum ve daha fazla bekleme anlamına gelir
- Büyük ölçekli ağ kesintileri quorum oluşturmayı imkânsız kılabilir

**Sloppy Quorum:** Bazı leaderless veritabanları, bir istemcinin çok sayıda replica'dan bağlantısı kesildiğinde bir quorum oluşturulmasını imkânsız kılacak şekilde, o key için olağan replica'lardan biri olmasa bile erişilebilir herhangi bir replica'nın yazımları kabul etmesine olanak tanır. Riak ve Dynamo buna **sloppy quorum** derken, Cassandra ve ScyllaDB bunu **consistency level ANY** olarak adlandırır. Sonraki okumaların yazılan değeri göreceğine dair bir garanti yoktur, ancak uygulamaya bağlı olarak yazımın başarısız olmasından daha iyi olabilir.

---
## 15. Concurrent Write Tespiti <a id='15-concurrent-writes'></a>

Multi-leader replikasyonda olduğu gibi, leaderless veritabanları da aynı key'e eşzamanlı yazımları kabul eder; bu, çözümlenmesi gereken çakışmalara yol açar.

Sorun şudur: Değişken ağ gecikmeleri ve kısmi arızalar nedeniyle olaylar farklı node'lara farklı sırayla ulaşabilir.

Her node, bir istemciden yazma isteği alındığında bir key için değerin üzerine yazsaydı, node'lar kalıcı olarak tutarsız hale gelirdi.

### Happens-Before İlişkisi ve Eşzamanlılık

İki işlemin eşzamanlı olup olmadığına nasıl karar veririz? Sezgiyi geliştirmek için bazı örneklere bakalım:

- Eğer A işlemi, B işleminin üzerine kurduğu bir değer ekliyorsa, A **happens before** (önce gerçekleşir) B. B, A'nın değerini kullanır ve A'dan sonra gerçekleşmiş olmalıdır.

- Öte yandan iki eşzamanlı yazımda, her istemci işlemi başlattığında diğer bir istemcinin aynı key üzerinde işlem yaptığını bilmez. Dolayısıyla işlemler arasında nedensel bir bağımlılık yoktur.

**Tanım:** A operasyonu B'yi biliyorsa, B'ye bağımlıysa veya herhangi bir şekilde B üzerine kuruyorsa, A **happens before** B olur. İki operasyonun eşzamanlı olup olmadığını belirlemek, eşzamanlılığın ne anlama geldiğini tanımlamanın anahtarıdır. İki operasyonun hiçbiri diğerinden önce gerçekleşmiyorsa, **concurrent** (eşzamanlı) olduğu söylenir.

İki A ve B operasyonu olduğunda üç olasılık vardır: ya A, B'den önce gerçekleşmiştir; ya B, A'dan önce gerçekleşmiştir; ya da A ve B eşzamanlıdır.

### Eşzamanlılık, Zaman ve Görelilik

İki işlem "aynı anda" gerçekleşiyorsa eşzamanlı olarak adlandırılmalıymış gibi görünebilir — ancak gerçekte, fiziksel olarak tam olarak aynı anda gerçekleşip gerçekleşmedikleri önemli değildir. Distributed sistemlerde saat sorunları nedeniyle, iki şeyin tam olarak aynı anda gerçekleşip gerçekleşmediğini söylemek oldukça zordur.

Eşzamanlılığı tanımlamak için, kesin zaman önemli değildir. İki operasyonu, birbirlerinden habersiz olmaları durumunda eşzamanlı olarak adlandırırız; fiziksel zaman nasıl olursa olsun. Bu bazen görelilik teorisiyle bağlantılı olarak değerlendirilir: ışıktan daha hızlı seyahat eden bir bilgi yoktur; dolayısıyla aralarındaki süre, mesafeyi ışığın kat etme süresinden kısa ise, belirli bir uzaklıkta meydana gelen iki olay birbirini etkileyemez.

### Happens-Before İlişkisini Yakalamak

İki operasyonun eşzamanlı mı yoksa birinin diğerinden önce mi gerçekleştiğini belirleyen bir algoritmaya bakalım. Basit tutmak için yalnızca bir replica'sı olan bir veritabanıyla başlayalım; ardından bu yaklaşımı leader'sız çok replica'lı bir veritabanına genelleştirebiliriz. Algoritma şu şekilde çalışır:

1. Sunucu her key için bir **version number** (versiyon numarası) tutar, o key her yazıldığında versiyon numarasını artırır ve yazılan değerle birlikte yeni versiyon numarasını saklar.

2. Bir istemci bir key'i okuduğunda, sunucu üzerine yazılmamış tüm değerleri (sibling'leri) ve en son versiyon numarasını döndürür. Bir istemci yazmadan önce bir key okumak zorundadır.

3. Bir istemci bir key'e yazarken, önceki okumadaki versiyon numarasını eklemeli ve önceki okumada aldığı tüm değerleri birleştirmelidir. Bir yazma isteğinden gelen yanıt da tüm sibling'leri döndürür; bu, birkaç yazımı zincirlememize olanak tanır.

4. Sunucu belirli bir versiyon numarasıyla bir yazma aldığında, o versiyon numarasına sahip veya daha küçük tüm değerlerin üzerine yazabilir (bunların yeni değerde birleştirildiğini bilir), ancak daha yüksek versiyon numarasına sahip tüm değerleri saklamalıdır (çünkü bu değerler gelen yazımla eşzamanlıdır).

**Alışveriş Sepeti Örneği (Şekil 6-15):**

1. Client 1, sepete `milk` ekler → versiyon 1 olarak kaydedilir
2. Client 2, `milk`ten habersiz `eggs` ekler → versiyon 2; sibling olarak hem `eggs` hem `milk` saklanır
3. Client 1, `flour` eklemek ister; server versiyon 1 değerinin üzerine yazar ama `eggs` sibling'ini saklar → versiyon 3 `[milk, flour]`; sibling: `[eggs]`
4. Client 2, önceki versiyonları (`milk` ve `eggs`) alıp birleştirerek `ham` ekler ve gönderir; server versiyon 2'nin üzerine yazar ama `[milk, flour]` sibling'ini saklar
5. Client 1, `bacon` eklemek ister; mevcut değerleri alıp birleştirir ve gönerir

### Version Vectors

Yukarıdaki örnek yalnızca tek bir replica kullandı. Birden fazla replica varken ancak leader yokken algoritma nasıl değişir?

Tek bir replica için tek bir versiyon numarası, operasyonlar arasındaki bağımlılıkları yakalamak için yeterliydi. Ancak birden fazla replica eşzamanlı yazımları kabul ederken, tek bir versiyon numarası yeterli değildir. Bunun yerine, her replica ve her key için bir versiyon numarasına ihtiyaç duyarız. Her replica, bir yazım işlerken kendi versiyon numarasını artırır ve ayrıca diğer replica'ların her birinden gördüğü versiyon numaralarını takip eder.

Tüm replica'lardan gelen versiyon numaralarının koleksiyonuna **version vector** (versiyon vektörü) denir. Version vector'ün en ilginç varyantı muhtemelen **dotted version vector**'dür ve Riak 2.0'da kullanılmaktadır.

Version vector'ler bazen **vector clock** (vektör saati) olarak da adlandırılır, ancak bunlar tam olarak aynı şey değildir. Replica'ların durumunu karşılaştırırken version vector'ler kullanılacak doğru veri yapısıdır.

Replica'ların durumunu kapsayan versiyon numaraları gibi, version vector'ler de değerler okunduğunda veritabanı replica'larından istemcilere gönderilir ve bir değer sonradan yazıldığında veritabanına geri gönderilmesi gerekir. Riak, version vector'ü **causal context** adı verdiği bir string olarak kodlar. Version vector, veritabanının üzerine yazmalar ile eşzamanlı yazımlar arasında ayrım yapmasına olanak tanır.

---
## 16. Özet ve Karşılaştırma <a id='16-ozet'></a>

Bu bölümde **replication** (replikasyon) konusunu inceledik. Replikasyon birçok amaca hizmet edebilir:

| Amaç | Açıklama |
|------|----------|
| **High Availability** | Bir makine (veya birkaç makine, bir zone veya hatta tüm bir region) kapansa bile sistemin çalışmaya devam etmesi |
| **Durability** | Tüm bir makine (hatta tüm bir region) kalıcı olarak arızalansa bile verilerin kaybolmaması |
| **Disconnected Operation** | Ağ kesintisine rağmen uygulamanın çalışmaya devam etmesi |
| **Latency** | Verilerin kullanıcılara coğrafi olarak yakın tutulması |
| **Scalability** | Tek bir makinenin kaldırabileceğinden daha yüksek hacimde okuma yükünün yönetilmesi |

### 3 Temel Replikasyon Yaklaşımı

| Yaklaşım | Açıklama | Avantaj | Dezavantaj |
|----------|----------|---------|------------|
| **Single-Leader** | İstemciler tüm yazımları tek bir node'a (leader) gönderir; leader bir veri değişikliği akışı diğer replica'lara (follower) gönderir | Anlaşılması kolay, güçlü tutarlılık | Tüm yazımlar tek node üzerinden geçmeli; leader arızalanırsa failover gerekli |
| **Multi-Leader** | İstemciler yazımlarını birkaç leader node'dan birine gönderir; liderler değişiklik akışlarını birbirlerine ve follower node'lara gönderir | Arızalı node, ağ kesintisi ve gecikme karşısında daha dirençli | Çakışma çözümü gerekli, zayıf tutarlılık |
| **Leaderless** | İstemciler her yazımı birkaç node'a gönderir ve stale veriyi düzeltmek için birden fazla node'dan paralel okuma yapar | Arızalı node, ağ kesintisi ve gecikme karşısında daha dirençli | Çakışma çözümü gerekli, zayıf tutarlılık |

### Replikasyon Lag Consistency Modelleri

| Model | Garanti |
|-------|--------|
| **Read-after-write consistency** | Kullanıcılar her zaman kendi gönderdikleri verileri görür |
| **Monotonic reads** | Kullanıcılar bir zaman noktasındaki verileri gördükten sonra daha önceki bir zaman noktasındaki verileri görmez |
| **Consistent prefix reads** | Kullanıcılar nedensel anlamda mantıklı bir durumdaki verileri görür; örneğin bir soruyu ve yanıtını doğru sırayla |

### Replikasyon Çakışma Çözüm Yöntemleri

| Yöntem | Açıklama | Uygun Durum |
|--------|----------|-------------|
| **LWW (Last Write Wins)** | En yüksek timestamp'li yazım kazanır | Veri kaybı kabul edilebilirse |
| **Manuel çözüm** | Tüm sibling değerler saklanır, uygulama veya kullanıcı birleştirir | Kullanıcı kararı gerekiyorsa |
| **CRDT** | Conflict-free replicated data types; her tür veri için otomatik birleştirme | Distributed veritabanları (Riak, Redis Enterprise) |
| **OT (Operational Transformation)** | İndeks tabanlı dönüştürme | Metin düzenleme (Google Docs) |

### Önemli Terimler Sözlüğü

In [ ]:
terms = {
    "Replication": "Aynı verinin kopyasını ağ üzerinden birden fazla makinede tutmak",
    "Replica": "Veritabanının bir kopyasını saklayan node",
    "Leader": "Yazım isteklerini kabul eden birincil node (primary, source olarak da bilinir)",
    "Follower": "Leader'dan veri değişikliklerini alıp uygulayan node (secondary, read replica, hot standby)",
    "Replication Log": "Leader'dan follower'lara gönderilen değişiklik kaydı",
    "Change Stream": "Replication log ile aynı anlam, veri değişikliklerinin akışı",
    "Failover": "Leader arızalandığında follower'lardan birinin leader olarak yükseltilmesi süreci",
    "Split Brain": "İki node'un eş zamanlı olarak leader olduğuna inanması; tehlikeli durum",
    "Fencing": "Split brain'e karşı eski leader'ı kısıtlama veya kapatma mekanizması",
    "WAL (Write-Ahead Log)": "Değişikliklerin önce yazıldığı, crash recovery için kullanılan log",
    "Statement-Based Replication": "SQL ifadelerinin follower'lara aynen iletilmesi yöntemi",
    "Logical (Row-Based) Replication": "Satır granülaritesinde değişikliklerin loglanması",
    "Binlog": "MySQL'in row-based replication için tuttuğu binary log",
    "CDC (Change Data Capture)": "Veritabanı değişikliklerini harici sistemlere aktarma tekniği",
    "Eventual Consistency": "Tüm replica'ların zamanla tutarlı hale geleceği garanti; 'nihai tutarlılık'",
    "Replication Lag": "Leader'daki yazım ile follower'a yansıması arasındaki gecikme",
    "Read-After-Write Consistency": "Kullanıcının kendi yazdığı veriyi okuyabilme garantisi",
    "Monotonic Reads": "Okumalar sırasında zamanın geriye gitmemesi garantisi",
    "Consistent Prefix Reads": "Nedensel olarak bağlı yazımların doğru sırada görülmesi garantisi",
    "Multi-Leader Replication": "Birden fazla node'un yazım kabul ettiği replikasyon; active/active veya bidirectional",
    "Geo-Distributed": "Birden fazla coğrafi region'a dağılmış sistem",
    "Replication Topology": "Yazımların node'lar arasında nasıl yayıldığını açıklayan iletişim yolları",
    "All-to-All Topology": "Her leader'ın diğer tüm leader'lara yazım gönderdiği topoloji",
    "Circular Topology": "Her node'un bir node'dan alıp bir node'a gönderdiği halka topoloji",
    "Star Topology": "Merkezi bir kök node'un diğer tüm node'lara yaydığı topoloji",
    "Sync Engine": "Offline-first ve local-first uygulamalarda veri senkronizasyonunu yöneten kütüphane",
    "Offline-First": "Çevrimdışıyken de çalışmaya devam edebilen uygulama",
    "Local-First Software": "Servis sağlayıcı kapatılsa bile çalışabilecek şekilde tasarlanmış işbirlikçi yazılım",
    "Conflict": "Farklı leader'larda aynı veriye eşzamanlı yapılan çakışan yazımlar",
    "LWW (Last Write Wins)": "En yüksek timestamp'e sahip yazımın kazandığı çakışma çözüm yöntemi",
    "Siblings": "Çakışma durumunda aynı key için eşzamanlı yazılan tüm değerler",
    "CRDT (Conflict-Free Replicated Data Type)": "Otomatik, deterministik çakışma çözümü sağlayan veri yapısı",
    "OT (Operational Transformation)": "Eşzamanlı metin düzenlemelerini birleştirmek için operasyonları dönüştürme yöntemi",
    "Strong Eventual Consistency": "Aynı yazım kümesini işlemiş tüm replica'ların aynı duruma gelmesi garantisi",
    "Leaderless Replication": "Herhangi bir replica'nın yazım kabul edebildiği; Dynamo-style veritabanları",
    "Dynamo-Style": "Amazon Dynamo'dan ilham alan leaderless replikasyon modeli",
    "Read Repair": "Okuma sırasında stale değerlerin tespit edilip güncellenmesi mekanizması",
    "Hinted Handoff": "Kullanılamayan node adına yazımların ipucu olarak saklanıp sonra iletilmesi",
    "Anti-Entropy": "Replica'lar arasındaki farklılıkları periyodik olarak arayan arka plan süreci",
    "Quorum": "Yazım/okuma başarısı için gereken minimum node onayı sayısı (w + r > n)",
    "Sloppy Quorum": "Normal quorum oluşturulamazsa herhangi erişilebilir node'un yazımı kabul etmesi",
    "Request Hedging": "Paralel istekler arasından en hızlı yanıtı kullanarak kuyruk gecikmesini azaltma",
    "Gray Failure": "Node'un tamamen çökmemesi ama anormal derecede yavaş çalışması durumu",
    "Happens-Before": "A operasyonu B'yi biliyorsa, B'ye bağımlıysa: A happens-before B",
    "Concurrent": "İki operasyonun birbirinden habersiz olması; ne A B'den önce ne de B A'dan önce",
    "Version Vector": "Tüm replica'lardan gelen versiyon numaralarının koleksiyonu; nedensel bağımlılık takibi",
    "Dotted Version Vector": "Riak 2.0'da kullanılan version vector varyantı",
    "Vector Clock": "Version vector ile benzer ancak birebir aynı olmayan kavram",
    "Causal Context": "Riak'ın version vector'ü kodladığı string",
    "Tiered Storage": "Sık ve seyrek erişilen verilerin farklı depolama katmanlarında tutulması",
    "Zero-Disk Architecture (ZDA)": "Tüm kalıcı verinin object storage'da tutulduğu, disklerin sadece cache için kullanıldığı mimari",
    "NewSQL": "Güçlü tutarlılık ve transaction desteğini distributed scalability ile birleştiren veritabanları",
    "Linearizability": "Tek node veritabanı gibi davranan güçlü tutarlılık modeli",
}

print(f"{'Terim':<45} {'Açıklama'}")
print("-" * 120)
for term, definition in terms.items():
    # Truncate long definitions for display
    disp_def = definition if len(definition) <= 75 else definition[:72] + "..."
    print(f"{term:<45} {disp_def}")

---

## Sonraki Bölüm

Bu bölüm, her replica'nın tüm veritabanının tam bir kopyasını sakladığını varsaydı; bu, büyük dataset'ler için gerçekçi değildir. Sonraki bölümde, her makinenin verinin yalnızca bir alt kümesini saklayabildiği **sharding** (bölümleme) konusuna bakacağız.

---

*Kaynak: Designing Data-Intensive Applications, Chapter 6: Replication*  
*Yazar: Martin Kleppmann*  
*Bu notebook, kitabın içeriğini Türkçe olarak özetleyip açıklamaktadır.*